In [2]:
import pandas as pd

patients = pd.read_csv("/Users/marcalbesa/Desktop/TFM/data/MIMM/patients_mimm.csv")
missing_trial_rows = int(patients["Trial"].isna().sum())
print(missing_trial_rows)

0


In [3]:

# Cargar CSVs
patients = pd.read_csv("/Users/marcalbesa/Desktop/TFM/data/MIMM/patients_mimm.csv")
df = pd.read_csv("/Users/marcalbesa/Desktop/TFM/data/MIMM/radiology_mimm.csv")

# Eliminar posible columna índice arrastrada del CSV de patients
patients = patients.loc[:, ~patients.columns.str.contains(r"^Unnamed")]
if "" in patients.columns:
    patients = patients.drop(columns=[""])

# Obtener el Trial de referencia por paciente desde patients
patients_trial = patients[["patient", "Trial"]].dropna(subset=["patient", "Trial"]).drop_duplicates()
trial_counts = patients_trial.groupby("patient")["Trial"].nunique()
conflicting_trials = trial_counts[trial_counts > 1]
if not conflicting_trials.empty:
    raise ValueError(
        "Hay pacientes con mas de un Trial en patients_mimm.csv: "
        f"{conflicting_trials.index.tolist()}"
    )
patients_trial = patients_trial.drop_duplicates(subset=["patient"], keep="first")

# Filtrar radiology para quedarnos solo con filas cuyo Trial coincide con el Trial del paciente en patients
before_rows = len(df)
df = df.merge(
    patients_trial.rename(columns={"Trial": "patients_trial"}),
    on="patient",
    how="left",
)
missing_trial_rows = int(df["patients_trial"].isna().sum())
df = df[df["Trial"] == df["patients_trial"]].copy()
removed_rows = before_rows - len(df)
print(f"Filas eliminadas por Trial no coincidente: {removed_rows}")
print(f"Filas de radiology sin Trial de referencia en patients: {missing_trial_rows}")
df = df.drop(columns=["patients_trial"])

# Crear grupos de pacientes consecutivos
# Cada vez que cambia el patient respecto a la fila anterior, empieza un grupo nuevo
df["group"] = (df["patient"] != df["patient"].shift()).cumsum()

# Quedarse solo con la primera fila de cada bloque consecutivo
collapsed = (
    df.groupby("group", as_index=False)
      .first()
)

# Eliminar la columna auxiliar si no la quieres en el resultado final
collapsed = collapsed.drop(columns="group")

# Guardar CSV final
collapsed.to_csv("archivo_colapsado.csv", index=False)

# Print de pacientes que siguen repetidos tras eliminar consecutivos
repeated_after = collapsed["patient"].value_counts()
repeated_after = repeated_after[repeated_after > 1]

print("\n=== PATIENT IDs REPETIDOS EN EL CSV FINAL ===")
print(repeated_after)

print("\n=== DATAFRAME FINAL ===")
collapsed


Filas eliminadas por Trial no coincidente: 2810
Filas de radiology sin Trial de referencia en patients: 2712

=== PATIENT IDs REPETIDOS EN EL CSV FINAL ===
patient
17261908    5
14748740    3
16398336    3
15563805    3
10012210    3
18900473    3
20810268    3
17280160    3
10584350    3
20416158    2
586698      2
14653930    2
11932685    2
12235737    2
11972396    2
13925217    2
14988315    2
15141193    2
20677961    2
11818229    2
12093400    2
12419489    2
16000430    2
20159853    2
20363213    2
374909      2
10651619    2
Name: count, dtype: int64

=== DATAFRAME FINAL ===


,patient,Trial,image_path,lesion_tag,pred_0,pred_1,pred_2,pred_3,pred_4,pred_5,...,pred_4086,pred_4087,pred_4088,pred_4089,pred_4090,pred_4091,pred_4092,pred_4093,pred_4094,pred_4095
0,20416158,Predict,Predict027_CT_BL_Abdomen_5.0_B20f_5_e1,LI1,2.868921,0.722953,0.376948,0.947282,0.592014,3.408894,...,0.671509,5.947426,2.059241,1.878515,1.332371,1.112355,3.563861,2.567007,3.324150,1.020617
1,10584350,Predict,Predict033_CT_BL_Abdomen_2.0_B30f_4_e1,LI1_bio,0.886173,0.854482,0.511731,0.161884,1.887517,0.000000,...,0.239164,0.753575,1.688230,0.889595,0.459008,1.997779,2.568159,1.758653,0.477091,0.083823
2,10651619,Predict,Predict037_CT_BL_Abdomen_2.0_B30f_4_e1,LI1,0.619327,1.064653,0.491814,0.163953,0.949436,0.313455,...,0.535010,1.531054,1.339161,1.566466,1.111939,1.204345,1.968053,1.460561,0.548578,0.245595
3,14748740,Predict,Predict038_CT_BL_CEV_abdomen_4_e1,A1,1.169351,1.059735,0.804578,0.569439,0.763403,1.525982,...,0.456098,2.511659,2.323280,1.410922,1.573610,1.416017,2.758127,1.243522,0.912137,0.294974
4,16471324,Predict,Predict020_CT_BL_1.25_TX_2_e1,LU1,0.000000,1.428203,0.364597,0.508201,1.918632,0.000000,...,0.000000,0.000000,0.204345,1.176408,1.350989,0.012821,3.483454,0.863846,0.000000,0.104254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,17215866,Immuno,17215866_20161128_4_Abdomen_5p0_B30f_3_B30f,LI3,0.305277,1.208927,0.961577,0.437853,1.029939,0.026684,...,0.296428,0.508750,0.626461,1.670361,0.733371,1.689983,1.852345,0.827108,0.034170,0.388982
315,18612626,Immuno,18612626_20170831_5_Abdomen_5p0_B30f_3_B30f,O1,2.469600,0.832108,0.942131,0.897415,0.918059,0.339772,...,0.758763,0.813853,2.491841,1.727048,0.599092,1.321230,2.700172,1.263605,0.428865,0.052899
316,18728195,Immuno,18728195_20151203_2_Abdomen_5p0_B30f_2_B30f,N1,0.274993,1.255128,0.976508,0.199296,1.757346,0.409406,...,0.610559,0.907990,1.416732,0.882677,0.989356,1.442335,2.105421,1.052126,0.577228,0.449474
317,18901461,Immuno,18901461_20180802_2_Abdomen_5p0_B30f_2_B30f,LI1,0.261357,1.029348,0.618621,0.000000,1.363442,0.049200,...,0.333588,0.937420,1.200290,1.516025,1.084780,1.733221,1.230740,0.710233,0.704670,0.138604


In [9]:
repeated_after

patient
17261908    5
14748740    3
16398336    3
15563805    3
10012210    3
18900473    3
20810268    3
17280160    3
10584350    3
20416158    2
586698      2
14653930    2
11932685    2
12235737    2
11972396    2
13925217    2
14988315    2
15141193    2
20677961    2
11818229    2
12093400    2
12419489    2
16000430    2
20159853    2
20363213    2
374909      2
10651619    2
Name: count, dtype: int64